# Kokoro-82M expressive fine-tune (Colab, resume-safe)
Goal: breathy/intimate, urgent/fearful and warm delivery with a stable speaker identity.
Run top to bottom. Every stage writes to Drive and skips work that is already done, so you can re-run after a disconnect.

**Runtime:** GPU (A100/L4 ideal, T4 works with the defaults below).
**Licences:** Expresso and NonverbalSpeech-38K are CC-BY-NC. Set `COMMERCIAL = True` to use only permissive sources.
**Your own audio:** put audio you own or are licensed to use in `MyDrive/kokoro_training/own/<speaker_id>/<emotion>/*.wav`.
Do not use ripped commercial dub audio.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ROOT = '/content/drive/MyDrive/kokoro_training'
COMMERCIAL = False            # True => drop CC-BY-NC datasets
USE_DEMUCS_ON_OWN = False     # only if your own audio has music/SFX under it
MAX_HOURS = {'expresso': 1.2, 'nonverbaltts': 1.0, 'libritts_r': 0.6, 'own': 6.0}
BUCKETS = {'soft': .35, 'intimate': .25, 'urgent': .20, 'warm': .20}
BUCKET_OF = {  # repo emotion tag -> bucket
    'neutral': 'soft', 'calm': 'soft', 'hesitant': 'soft', 'monotone': 'soft',
    'whisper': 'intimate', 'sad': 'intimate', 'sigh': 'intimate', 'cry': 'intimate',
    'fearful': 'urgent', 'nervous': 'urgent', 'gasp': 'urgent', 'angry': 'urgent', 'shout': 'urgent',
    'happy': 'warm', 'excited': 'warm', 'laugh': 'warm', 'confident': 'warm', 'dramatic': 'warm',
}
# Expresso style -> repo tag (verify names against ds.features['style'].names)
EXPRESSO_MAP = {'default': 'neutral', 'happy': 'happy', 'sad': 'sad', 'whisper': 'whisper',
                'confused': 'hesitant', 'laughing': 'laugh', 'enunciated': 'neutral',
                'essentials': 'neutral', 'singing': None, 'longform': 'neutral', 'emphasis': 'dramatic'}
import os
for d in ['raw', 'clips', 'manifests', 'ckpt/stage1', 'ckpt/stage2', 'out']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

In [ ]:
!pip install -q datasets soundfile librosa pyloudnorm faster-whisper phonemizer misaki onnx onnxruntime pandas demucs
!apt-get -qq install -y espeak-ng
!test -d /content/kikiri-tts || git clone -q https://github.com/semidark/kikiri-tts.git /content/kikiri-tts
!test -d /content/kokoro-tts || echo "Upload/clone this repo's training/ folder to /content/kokoro-tts/training"

## 1. Pull public data (streaming, capped by hours)
Column names differ per dataset; the loaders print the schema first. Adjust `text_col` if a dataset changes.

In [ ]:
import soundfile as sf, numpy as np, pandas as pd, librosa, itertools
from datasets import load_dataset
rows = []
def dump(ds, name, text_col, label_fn, max_h, spk_col=None):
    out = f'{ROOT}/raw/{name}'; os.makedirs(out, exist_ok=True); secs = 0
    for i, ex in enumerate(ds):
        lab = label_fn(ex)
        if lab is None: continue
        a = ex['audio']; y, sr = np.asarray(a['array'], dtype='float32'), a['sampling_rate']
        fn = f'{out}/{name}_{i:06d}.wav'
        if not os.path.exists(fn): sf.write(fn, y, sr)
        rows.append(dict(path=fn, transcript=ex[text_col], speaker_id=f"{name}_{ex[spk_col]}" if spk_col else name,
                         emotion=lab, source=name))
        secs += len(y)/sr
        if secs/3600 >= max_h: break
    print(name, round(secs/3600, 2), 'h')

if not COMMERCIAL:
    ds = load_dataset('ylacombe/expresso', split='train', streaming=True)
    print(next(iter(ds)).keys())
    names = ds.features['style'].names if hasattr(ds.features.get('style'), 'names') else None
    lab = lambda ex: EXPRESSO_MAP.get(names[ex['style']] if names else ex['style'])
    dump(ds, 'expresso', 'text', lab, MAX_HOURS['expresso'], 'speaker_id')

ds = load_dataset('deepvk/NonverbalTTS', split='train', streaming=True)
print(next(iter(ds)).keys())   # inspect: keep rows whose text contains NV tags / emotion labels

**Check the NonverbalTTS schema printed above**, then adapt the `label_fn` below (it maps its emotion label to repo tags and
rewrites NV tags to `[breath]`, `[gasp]`, `[laugh]`, `[sigh]`). Defaults assume columns `text` and `emotion`.

In [ ]:
import re
NV_MAP = {'breathing': '[breath]', 'breath': '[breath]', 'laughter': '[laugh]', 'sigh': '[sigh]', 'cough': None}
EMO_MAP = {'happy': 'happy', 'sad': 'sad', 'angry': 'angry', 'fear': 'fearful', 'neutral': 'neutral', 'surprise': 'excited', 'disgust': None}
lab = lambda ex: EMO_MAP.get(str(ex.get('emotion', 'neutral')).lower())
ds = load_dataset('deepvk/NonverbalTTS', split='train', streaming=True)
dump(ds, 'nonverbaltts', 'text', lab, MAX_HOURS['nonverbaltts'])

ds = load_dataset('mythicinfinity/libritts_r', 'clean', split='train.clean.100', streaming=True)
dump(ds, 'libritts_r', 'text_normalized', lambda ex: 'neutral', MAX_HOURS['libritts_r'], 'speaker_id')
pd.DataFrame(rows).to_csv(f'{ROOT}/raw/public_meta.csv', index=False)

## 2. Your own audio (optional but this is what locks identity)
Layout: `own/<speaker_id>/<emotion>/*.wav`, where `<emotion>` is a repo tag (`whisper`, `fearful`, `happy`, ...).
Transcripts are generated by Whisper in step 3. Archetype suggestions per speaker are in the guide.

In [ ]:
import glob, subprocess
own = []
for p in glob.glob(f'{ROOT}/own/*/*/*.*'):
    spk, emo = p.split('/')[-3], p.split('/')[-2]
    own.append(dict(path=p, speaker_id=spk, emotion=emo, source='own'))
print(len(own), 'own files')
if USE_DEMUCS_ON_OWN and own:
    subprocess.run(['python', '-m', 'demucs', '--two-stems=vocals', '-o', f'{ROOT}/raw/demucs', *[o['path'] for o in own]])
    for o in own:
        o['path'] = f"{ROOT}/raw/demucs/htdemucs/{os.path.splitext(os.path.basename(o['path']))[0]}/vocals.wav"

## 3. Slice on breath boundaries, normalise, transcribe
2.5-8 s clips, cut at pauses (keeping 150 ms so breaths survive), 24 kHz mono, -23 LUFS, peak 0.95.

In [ ]:
import pyloudnorm as pyln
from faster_whisper import WhisperModel
SR = 24000; meter = pyln.Meter(SR)
whisper = WhisperModel('large-v3', device='cuda', compute_type='float16')

def slice_file(path, min_s=2.5, max_s=8.0, keep=0.15, top_db=38):
    y, _ = librosa.load(path, sr=SR, mono=True)
    iv = librosa.effects.split(y, top_db=top_db, frame_length=2048, hop_length=512)
    pad = int(keep*SR); out = []; cur = None
    for s, e in iv:
        s, e = max(0, s-pad), min(len(y), e+pad)
        if cur is None: cur = [s, e]; continue
        if s - cur[1] < 0.35*SR and (e-cur[0])/SR <= max_s: cur[1] = e
        else: out.append(cur); cur = [s, e]
    if cur: out.append(cur)
    for s, e in out:
        if min_s <= (e-s)/SR <= max_s: yield y[s:e]

def norm(y):
    l = meter.integrated_loudness(y)
    y = pyln.normalize.loudness(y, l, -23.0)
    pk = np.abs(y).max()
    return y*(0.95/pk) if pk > 0.95 else y

pub = pd.read_csv(f'{ROOT}/raw/public_meta.csv').to_dict('records')
meta_path = f'{ROOT}/clips/metadata.csv'
done = pd.read_csv(meta_path) if os.path.exists(meta_path) else pd.DataFrame()
seen = set(done['src'].tolist()) if len(done) else set()
out_rows = done.to_dict('records')
for r in pub + own:
    if r['path'] in seen: continue
    for k, seg in enumerate(slice_file(r['path'])):
        seg = norm(seg)
        fn = f"{ROOT}/clips/{r['source']}_{abs(hash(r['path']))%10**8}_{k}.wav"
        sf.write(fn, seg, SR)
        text = r.get('transcript')
        if r['source'] != 'own' and isinstance(text, str) and r['source'] != 'nonverbaltts' and k == 0:
            pass  # public transcripts cover the whole utterance, not slices; re-transcribe for accuracy
        segs, _ = whisper.transcribe(fn, language='en', beam_size=5, vad_filter=False)
        text = ' '.join(s.text.strip() for s in segs).strip()
        if len(text) < 4: os.remove(fn); continue
        out_rows.append(dict(filename=os.path.basename(fn), transcript=text, speaker_id=r['speaker_id'],
                             emotion=r['emotion'], source=r['source'], src=r['path']))
    seen.add(r['path'])
    pd.DataFrame(out_rows).to_csv(meta_path, index=False)   # checkpoint per file
print(len(out_rows), 'clips')

## 4. Balance to 35/25/20/20 (max 2x oversampling), then phonemize
Whisper drops breaths from text. Insert `[breath]` at the start of clips whose first 150 ms is breath-like if you want explicit tokens
(see guide step 5); the repo phonemizer preserves `[breath]`/`[gasp]` when present.

In [ ]:
df = pd.read_csv(meta_path)
df['bucket'] = df['emotion'].map(BUCKET_OF)
df = df.dropna(subset=['bucket'])
dur = lambda f: sf.info(f'{ROOT}/clips/{f}').duration
df['dur'] = df['filename'].map(dur)
have = df.groupby('bucket')['dur'].sum()/3600
print(have)
total = have.sum()
parts = []
for b, frac in BUCKETS.items():
    sub = df[df.bucket == b]
    if sub.empty: print('MISSING bucket', b); continue
    need = frac*total; got = sub['dur'].sum()/3600
    reps = min(2.0, max(1.0, need/got)) if got else 1
    n = int(len(sub)*reps)
    parts.append(sub.sample(n, replace=reps > 1, random_state=0))
bal = pd.concat(parts).drop(columns=['src','bucket','dur'])
bal.to_csv(f'{ROOT}/manifests/metadata.csv', index=False)
print(len(bal), 'rows; sources:', bal.source.value_counts().to_dict())

In [ ]:
!cd /content/kokoro-tts && python training/phonemize_dataset.py \
  --csv {ROOT}/manifests/metadata.csv --wav_dir {ROOT}/clips --output_dir {ROOT}/manifests --val_ratio 0.05
!head -3 {ROOT}/manifests/train_list.txt; wc -l {ROOT}/manifests/*.txt

## 5. Patch the upstream harness (once)
Apply the loader patch described in `training/README.md`: read `conditioning.jsonl`, feed duration targets + emotion id, add
`0.1 * style_classification_loss`. Then set precision/accumulation below.

In [ ]:
import yaml, torch
cfgp = '/content/kikiri-tts/Configs/config.yml'
cfg = yaml.safe_load(open(cfgp))
vram = torch.cuda.get_device_properties(0).total_memory/2**30
big = vram > 30
cfg['batch_size'] = 8 if big else 4
cfg['max_len'] = 400 if big else 300
cfg['accum_steps'] = 1 if big else 2
cfg['mixed_precision'] = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
yaml.safe_dump(cfg, open(cfgp, 'w'))
print(vram, cfg['batch_size'], cfg['accum_steps'], cfg['mixed_precision'])

## 6. Train (auto-resume). Stop Stage 1 when val mel loss is about 0.35-0.40.

In [ ]:
import glob
%cd /content/kikiri-tts
!test -f models/kokoro-v1_0.pth || python -c "from huggingface_hub import hf_hub_download as h; h('hexgrad/Kokoro-82M','kokoro-v1_0.pth',local_dir='models')"
last1 = sorted(glob.glob(f'{ROOT}/ckpt/stage1/*.pth'))
resume1 = f'--pretrained_model {last1[-1]}' if last1 else '--pretrained_model models/kokoro-v1_0.pth'
!python train_first.py --config_path Configs/config.yml --train_data {ROOT}/manifests/train_list.txt \
  --val_data {ROOT}/manifests/val_list.txt {resume1} --save_dir {ROOT}/ckpt/stage1

In [ ]:
last2 = sorted(glob.glob(f'{ROOT}/ckpt/stage2/*.pth'))
src2 = last2[-1] if last2 else f'{ROOT}/ckpt/stage1/best_model.pth'
# Stage 2 LRs: generator 1e-5, discriminator 5e-5 (set in config)
!python train_second.py --config_path Configs/config.yml --train_data {ROOT}/manifests/train_list.txt \
  --val_data {ROOT}/manifests/val_list.txt --pretrained_model {src2} --save_dir {ROOT}/ckpt/stage2

## 7. Voicepacks + INT8 ONNX export

In [ ]:
!cd /content/kokoro-tts && python training/extract_voicepacks.py --out_dir {ROOT}/out/emotions
!cd /content/kokoro-tts && python training/export_onnx.py --model {ROOT}/ckpt/stage2/best_model.pth \
  --out_dir {ROOT}/out/models --conditioning_nodes style_projection,duration_predictor,emotion_embedding
import numpy as np, glob
for f in glob.glob(f'{ROOT}/out/emotions/*.npy'):
    a = np.load(f); assert a.shape == (510, 1, 256) and a.dtype == np.float32, (f, a.shape, a.dtype)
print('voicepacks OK')